# ============================================================
# SHIP 5: IBM GRANITE AGENTIC RAG WITH EVIDENCEFLOW VERIFICATION
# ============================================================

## Credits & Attribution

**Foundation:**
This notebook is built on the foundational structure and methods learned from the IBM SkillsBuild lab: *"Build a LangChain agentic RAG system using the Granite-4-H-Small model in watsonx.ai."*

Original lab authored by **Anna Gutowska**. IBM SkillsBuild, 2026.

**Inspiration:**
The evidence verification layer (evidence IDs, citation verification, fail-closed behavior) was inspired by **Asaif Ali's EvidenceFlow** project.

🔗 https://github.com/AsaifAli/EvidenceFlow

---

## My Additions

- **EvidenceFlow verification layer** — evidence IDs, citation verification, fail-closed behavior
- **Local sovereign execution** — Ollama, no watsonx.ai cloud dependency
- **Genealogy-specific adaptations** — partial names, phonetic matching
- **Containerized "Mirror of Becoming" output format** — verified, complete archive

---

## My Setup

- All models run locally on my M1 MacBook Air
- No cloud dependencies — sovereign by design
- Built for: A Mirror of My Becoming — Sovereign personal archive
- Author: Evelyn Caro (@qaevelyn)
- Date: 2026-09-02
- LLM: IBM Granite 4.1:3b (via Ollama)
- Type: Agentic RAG with EvidenceFlow verification

In [1]:
# ============================================================
# CELL 1: SET UP ENVIRONMENT AND LOAD DEPENDENCIES
# ============================================================

get_ipython().system('echo "::group::Install Dependencies"')
get_ipython().system('%pip install uv')
get_ipython().system('uv pip install "git+https://github.com/ibm-granite-community/utils.git" langchain_core langchain_classic langchain_community langchain_text_splitters langchain_ollama chromadb tiktoken bs4')
get_ipython().system('echo "::endgroup::"')

print("✅ Dependencies installed")

::group::Install Dependencies
zsh:fg:1: no job control in this shell.
Using Python 3.13.5 environment at: /Users/evelyn/Documents/Mirror-Project/granite-workshop/.venv
Resolved 108 packages in 2.96s                                       
Checked 108 packages in 6ms
::endgroup::
✅ Dependencies installed


In [2]:
# ============================================================
# CELL 2: IMPORTS
# ============================================================

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.tools import tool
from langchain_classic.tools.render import render_text_description_and_args
from langchain_classic.agents.output_parsers import JSONAgentOutputParser
from langchain_classic.agents.format_scratchpad import format_log_to_str
from langchain_classic.agents import AgentExecutor
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.runnables import RunnablePassthrough

print("✅ Imports ready")

/var/folders/4l/746jmy4j7b3711l8cgn7h1fm0000gn/T/ipykernel_10753/4132297640.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
/Users/evelyn/Documents/Mirror-Project/granite-workshop/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
USER_AGENT environment variable not set, consider setting it to identify your requests.


✅ Imports ready


In [3]:
# ============================================================
# CELL 3: DEFINE RAG TOOL AND INITIALIZE AGENT
# ============================================================

# This step establishes a baseline. By running the agent without tools first,
# we can see what the LLM generates on its own — then compare it to the
# tool-enhanced version later.

# Parameters:
# - Model: granite4.1:3b (via Ollama)
# - Temperature: 0.7
# - Stop sequences: Observation, Human: (to prevent hallucinations)

# Why this matters:
# - The agent needs to know when to stop generating
# - Without stop sequences, it may continue past the desired output
# - With tools, the agent can retrieve real evidence instead of guessing

# My setup:
# - All models run locally on my M1 MacBook Air
# - No cloud dependencies — sovereign by design

print("✅ Cell 3: Define RAG Tool and Initialize Agent ready")

✅ Cell 3: Define RAG Tool and Initialize Agent ready


In [4]:
# ============================================================
# CELL 4: SET UP MODEL (IBM Granite)
# ============================================================

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="granite4.1:3b",
    temperature=0.7,
)

print("✅ Model ready with granite4.1:3b")

✅ Model ready with granite4.1:3b


In [5]:
# ============================================================
# CELL 5: SET UP EMBEDDINGS
# ============================================================

from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
)

print("✅ Embeddings ready with nomic-embed-text")

✅ Embeddings ready with nomic-embed-text


In [6]:
# ============================================================
# CELL 6: SET UP PROMPT TEMPLATE
# ============================================================

from langchain_core.prompts import ChatPromptTemplate

# Simple prompt template for querying the model
template = "Answer the {query} accurately. If you do not know the answer, simply say you do not know."
prompt = ChatPromptTemplate.from_template(template)

print("✅ Prompt template ready")

✅ Prompt template ready


In [7]:
# ============================================================
# CELL 7: SET UP THE CHAIN
# ============================================================

from langchain_core.output_parsers import StrOutputParser

# Set up a chain with the prompt and LLM
# This allows the generative model to produce a response
agent = prompt | llm | StrOutputParser()

print("✅ Chain ready")

✅ Chain ready


In [8]:
# ============================================================
# CELL 8: EVIDENCEFLOW VERIFICATION LAYER
# ============================================================

import uuid
from datetime import datetime

# Evidence Registry
evidence_registry = {}

def assign_evidence_id(chunk, source):
    """Assign a unique evidence ID to a chunk."""
    timestamp = datetime.now().strftime("%Y-%m-%d-%H%M%S")
    uid = str(uuid.uuid4())[:8]
    evidence_id = f"EVI-{timestamp}-{uid}"
    evidence_registry[evidence_id] = {
        "chunk": chunk,
        "source": source,
        "timestamp": timestamp
    }
    return evidence_id

def generate_citation(evidence_id):
    """Generate a citation for a given evidence ID."""
    return f"[{evidence_id}]"

def verify_claims(answer, evidence_ids):
    """Check that each claim in the answer has a corresponding evidence ID."""
    for eid in evidence_ids:
        if eid not in evidence_registry:
            return False, f"Evidence ID {eid} not found in registry"
    return True, "All evidence verified"

def answer_with_verification(query, retriever, llm):
    """Retrieve, generate, and verify before returning."""
    # Retrieve
    docs = retriever.get_relevant_documents(query)
    
    # Assign evidence IDs
    evidence_ids = []
    for doc in docs:
        eid = assign_evidence_id(doc.page_content, doc.metadata.get("source", "unknown"))
        evidence_ids.append(eid)
    
    # Generate answer
    context = "\n\n".join([doc.page_content for doc in docs])
    prompt_text = f"Based on the following context, answer the query.\n\nContext: {context}\n\nQuery: {query}\n\nAnswer:"
    response = llm.invoke(prompt_text)
    
    # Verify
    verified, message = verify_claims(response.content, evidence_ids)
    if not verified:
        return f"⚠️ Cannot verify answer. {message}"
    
    # Add citations
    citations = " ".join([generate_citation(eid) for eid in evidence_ids])
    return f"{response.content}\n\n{'-'*40}\n📎 Citations: {citations}"

print("✅ EvidenceFlow verification layer ready")

✅ EvidenceFlow verification layer ready


In [12]:
# ============================================================
# CELL 9: TEST THE AGENT (Forced Tool Use)
# ============================================================

response = agent.invoke({
    "query": "Using the retriever tool, find information about 'A Mirror of My Becoming'."
})

print("=" * 60)
print("QUESTION: What is A Mirror of My Becoming?")
print("=" * 60)
print(response)

QUESTION: What is A Mirror of My Becoming?
I do not know the answer.


In [ ]:
# ============================================================
# CELL 10: SOVEREIGNTY QUERY
# ============================================================

response = agent_executor.invoke({
    "input": "How does A Mirror of My Becoming use AI and sovereignty?"
})

print("=" * 60)
print("QUESTION: How does A Mirror of My Becoming use AI and sovereignty?")
print("=" * 60)
print(response["output"])

In [ ]:
# ============================================================
# CELL 11: ESTABLISH KNOWLEDGE BASE AND RETRIEVER
# ============================================================

import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ------------------------------------------------------------
# LOCAL DOCUMENTS (Primary)
# ------------------------------------------------------------

file_paths = [
    "/Users/evelyn/Documents/Mirror-Project/MIRROR_LOG.md",
    "/Users/evelyn/Documents/Mirror-Project/md files/PER_SCHOLAS_LOG_v2.md",
    "/Users/evelyn/Documents/Mirror-Project/md files/MIRROR_LOG_v2.md",
]

documents = []
for file_path in file_paths:
    if os.path.exists(file_path):
        print(f"✅ Loading: {file_path}")
        loader = TextLoader(file_path)
        documents.extend(loader.load())
    else:
        print(f"❌ File not found: {file_path}")

# ------------------------------------------------------------
# DEEPSEEK CONVERSATIONS (Secondary)
# ------------------------------------------------------------

deepseek_path = "/Users/evelyn/Documents/Mirror-Project/conversations/conversations.json"

if os.path.exists(deepseek_path):
    print(f"✅ Loading DeepSeek conversations: {deepseek_path}")
    loader = TextLoader(deepseek_path)
    documents.extend(loader.load())
else:
    print(f"⚠️ DeepSeek conversations not found at: {deepseek_path}")

# ------------------------------------------------------------
# OPTIONAL: WEB SOURCES (For 10 Million Names)
# ------------------------------------------------------------

# Uncomment and add URLs when needed
# urls = [
#     "https://example.com/10-million-names-data",
#     "https://example.com/census-records",
# ]
# 
# from langchain_community.document_loaders import WebBaseLoader
# docs = [WebBaseLoader(url).load() for url in urls]
# docs_list = [item for sublist in docs for item in sublist]
# documents.extend(docs_list)
# print(f"✅ Loaded {len(docs_list)} documents from web")

# ------------------------------------------------------------
# CHUNK AND CREATE VECTOR STORE
# ------------------------------------------------------------

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""],
)

texts = text_splitter.split_documents(documents)

print(f"✅ Created {len(texts)} chunks from {len(file_paths)} files + DeepSeek conversations")

# Create or connect to vector store
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print("✅ Knowledge base and retriever established")

In [ ]:
# ============================================================
# CELL 12: YOUR WORKING RETRIEVAL CODE (From Ship 3/4)
# ============================================================

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma

# Set up Ollama
llm = ChatOllama(model="granite4.1:3b")
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Connect to your ChromaDB
vector_store = Chroma(
    collection_name="deepseek_conversations",
    embedding_function=embeddings,
    persist_directory="/Users/evelyn/Documents/Mirror-Project/vector_db"
)

# Create retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

# Test it
print("=" * 60)
print("TEST RETRIEVAL: What is A Mirror of My Becoming?")
print("=" * 60)

results = retriever.get_relevant_documents("What is A Mirror of My Becoming?")

print(f"Found {len(results)} relevant chunks:")
for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:300] + "..." if len(doc.page_content) > 300 else doc.page_content)

print("\n✅ Retrieval test complete")

In [16]:
# ============================================================
# CELL 13: CHUNKING WITH TIKTOKEN (Optimized)
# ============================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,      # Increased from 250
    chunk_overlap=50,    # Added overlap
)

doc_splits = text_splitter.split_documents(documents)

print(f"✅ Created {len(doc_splits)} chunks using tiktoken")

NameError: name 'documents' is not defined

In [17]:
# ============================================================
# CELL 14: CREATE VECTOR STORE
# ============================================================

from langchain_chroma import Chroma

# Store embedded documents in Chroma DB
vectorstore = Chroma.from_documents(
    documents=doc_splits,
    collection_name="agentic-rag-chroma",
    embedding=embeddings,
)

print(f"✅ Vector store created with {len(doc_splits)} chunks")

NameError: name 'doc_splits' is not defined

In [19]:
# ============================================================
# CELL 15: SET UP RETRIEVER
# ============================================================

retriever = vectorstore.as_retriever()

print("✅ Retriever ready")

NameError: name 'vectorstore' is not defined

# ============================================================
# CELL 16: DEFINE RAG TOOL AND INITIALIZE AGENT
# ============================================================

**Step 3: Initialize a basic agent with no tools.**

This step establishes a baseline. By running the agent without tools first, we can see what the LLM generates on its own — then compare it to the tool-enhanced version later.

**Parameters:**
- Model: `granite4.1:3b` (via Ollama)
- Temperature: 0.7
- Stop sequences: `Observation`, `Human:` (to prevent hallucinations)

**Why this matters:**
- The agent needs to know when to stop generating
- Without stop sequences, it may continue past the desired output
- With tools, the agent can retrieve real evidence instead of guessing

**My setup:**
- All models run locally on my M1 MacBook Air
- No cloud dependencies — sovereign by design

In [18]:
# ============================================================
# CELL 17: RAG TOOL DEFINITION
# ============================================================

from langchain_classic.tools import tool

@tool
def get_mirror_context(question: str):
    """Retrieve information from A Mirror of My Becoming project documents."""
    context = retriever.invoke(question)
    return context

tools = [get_mirror_context]

print("✅ RAG tool defined")

✅ RAG tool defined


In [ ]:
# ============================================================
# CELL 18: ESTABLISH PROMPT TEMPLATE
# ============================================================

**Step 4: Set up a structured chat prompt for the agent.**

This template is more complex. It is a structured chat prompt used for agents with multiple tools. In our case, the tool was defined in Step 17.

The structured chat prompt consists of:
- A **system_prompt** — instructs the agent to print its "thought process," including subtasks, tools used, and final output. This gives insight into the agent's function calling.
- A **human_prompt** — the user's question.
- The **RAG tool** — defined in Step 17.

**Why this matters:**
- The agent needs to show its reasoning so we can verify its decisions
- JSON Blob format ensures structured, parseable output
- This is how the agent decides when to use the RAG tool vs. when to answer directly

**My setup:**
- All models run locally on my M1 MacBook Air
- No cloud dependencies — sovereign by design

In [ ]:
# ============================================================
# CELL 19: SYSTEM PROMPT
# ============================================================

system_prompt = """Respond to the human as helpfully and accurately as possible. You have access to the following tools: {tools}
Use a json blob to specify a tool by providing an action key (tool name) and an action_input key (tool input).
Valid "action" values: "Final Answer" or {tool_names}
Provide only ONE action per $JSON_BLOB, as shown:

{{ 
  "action": $TOOL_NAME,
  "action_input": $INPUT
}}

Follow this format:
Question: input question to answer
Thought: consider previous and subsequent steps
Action:
$JSON_BLOB
Observation: action result
... (repeat Thought/Action/Observation N times)
Thought: I know what to respond
Action:
{{ 
  "action": "Final Answer",
  "action_input": "Final response to human"
}}

Begin! Reminder to ALWAYS respond with a valid json blob of a single action.
Respond directly if appropriate. Format is Action: $JSON_BLOB then Observation"""

print("✅ System prompt defined")

In [ ]:
# ============================================================
# CELL 20: HUMAN PROMPT
# ============================================================

human_prompt = """{input}
{agent_scratchpad}
(reminder to always respond in a JSON blob)"""

print("✅ Human prompt defined")

In [ ]:
# ============================================================
# CELL 21: COMBINE PROMPTS
# ============================================================

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Combine system and human prompts into a structured chat prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", human_prompt),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

print("✅ Combined prompt ready")

In [ ]:
# ============================================================
# CELL 22: PROMPT TEMPLATE WITH MEMORY
# ============================================================

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Establish the order of prompts in the template
# This creates a template with:
# 1. system_prompt — instructions for the agent
# 2. chat_history — optional memory from previous interactions
# 3. human_prompt — user input + agent_scratchpad
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", human_prompt),
    ]
)

print("✅ Prompt template with memory ready")

In [ ]:
# ============================================================
# CELL 23: FINALIZE PROMPT TEMPLATE
# ============================================================

from langchain_classic.tools.render import render_text_description_and_args

# Finalize the prompt template by adding tool names, descriptions, and arguments
# This allows the agent to access information about each tool, including intended use cases
# It also means we can add/remove tools without altering the entire prompt template
prompt = prompt.partial(
    tools=render_text_description_and_args(list(tools)),
    tool_names=", ".join([t.name for t in tools]),
)

print("✅ Prompt template finalized with tool info")

In [ ]:
# ============================================================
# CELL 24: SET UP AGENT MEMORY (CHECKPOINTER)
# ============================================================

from langgraph.checkpoint.memory import InMemorySaver

# Set up short-term memory using a checkpointer
memory = InMemorySaver()

print("✅ Agent memory (checkpointer) ready")

In [ ]:
# ============================================================
# CELL 25: SET UP AGENT CHAIN AND EXECUTOR (Corrected)
# ============================================================

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Set up memory
memory = InMemorySaver()

# Create the agent using the modern API
# The prompt is passed via system_prompt parameter
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=memory,
)

print("✅ Agent created with memory")

In [ ]:
# ============================================================
# CELL 26: GENERATE RESPONSE (RAG Pipeline Query)
# ============================================================

response = agent_executor.invoke({
    "input": "What are the RAG pipelines in the Mirror fleet?"
})

print("=" * 60)
print("QUESTION: What are the RAG pipelines in the Mirror fleet?")
print("=" * 60)
print(response["output"])

In [ ]:
# ============================================================
# CELL 28: GENERATE RESPONSE (Ida B. Wells Standard Query)
# ============================================================

response = agent_executor.invoke({
    "input": "What is the Ida B. Wells Standard of record keeping?"
})

print("=" * 60)
print("QUESTION: What is the Ida B. Wells Standard of record keeping?")
print("=" * 60)
print(response["output"])

In [ ]:
# ============================================================
# CELL 29: GENERATE RESPONSE (EvidenceFlow Query)
# ============================================================

response = agent_executor.invoke({
    "input": "What is the EvidenceFlow verification layer?"
})

print("=" * 60)
print("QUESTION: What is the EvidenceFlow verification layer?")
print("=" * 60)
print(response["output"])

In [13]:
# Quick test: Does the retriever find anything?
test_query = "A Mirror of My Becoming"
results = retriever.invoke(test_query)

print("=" * 60)
print(f"TEST RETRIEVAL: '{test_query}'")
print("=" * 60)
print(f"Found {len(results)} chunks")
for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:300] + "..." if len(doc.page_content) > 300 else doc.page_content)

NameError: name 'retriever' is not defined